# LLM Inference Hub — Colab

Cloud GPU inference with Gradio UI + public link.

**Usage:** Edit cell 1 config → Run all cells.

In [ ]:
#@title 1. Config
MODEL_PATH  = "/content/drive/MyDrive/path/to/model"  #@param {type:"string"}
USE_4BIT    = False  #@param {type:"boolean"}
PORT        = 8000
HOST        = "0.0.0.0"

import os
print(f"Model: {MODEL_PATH}")
print(f"4-bit: {USE_4BIT}")

In [ ]:
#@title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os
print("OK" if os.path.isdir(MODEL_PATH) else f"NOT FOUND: {MODEL_PATH}")

In [ ]:
#@title 3. Install dependencies
import importlib, subprocess, sys
for pkg in ['transformers', 'accelerate', 'huggingface_hub', 'sentencepiece', 'fastapi', 'uvicorn', 'gradio']:
    try:
        importlib.import_module(pkg.replace('-','_'))
        print(f"  {pkg} ✓")
    except ImportError:
        print(f"  {pkg} installing...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])
print("Done.")

In [ ]:
#@title 4. Load model
import torch, time, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB)")

quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type='nf4') if USE_4BIT else None
load_kwargs = dict(torch_dtype=torch.bfloat16 if not USE_4BIT else None, device_map='auto', trust_remote_code=True)
if quant_config:
    load_kwargs['quantization_config'] = quant_config

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, **load_kwargs)
model.eval()

vram = torch.cuda.memory_allocated()/1e9 if torch.cuda.is_available() else 0
print(f"Loaded in {time.time()-t0:.0f}s | {sum(p.numel() for p in model.parameters())/1e9:.1f}B params | VRAM: {vram:.1f} GB")

In [ ]:
#@title 5. Quick test
msgs = [{"role": "user", "content": "Say hello."}]
text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=64, temperature=0.7, do_sample=True)
print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

In [ ]:
#@title 6. Start FastAPI server
import threading, torch, gc
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from typing import Optional, List
import uvicorn

def gen(msgs, max_tokens=512, temp=0.7, top_p=0.9, top_k=50, rep=1.0):
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tokenizer(text, return_tensors='pt').to(model.device)
    kw = dict(max_new_tokens=max_tokens, repetition_penalty=rep)
    if temp > 0: kw.update(dict(temperature=temp, do_sample=True, top_p=top_p, top_k=top_k))
    else: kw['do_sample'] = False
    with torch.no_grad():
        out = model.generate(**inp, **kw)
    return tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()

app = FastAPI()
class M(BaseModel):
    role: str; content: str
class CR(BaseModel):
    model: Optional[str] = 'model'; messages: List[M]
    temperature: Optional[float] = 0.7; top_p: Optional[float] = 0.9
    top_k: Optional[int] = 50; max_tokens: Optional[int] = 512; repetition_penalty: Optional[float] = 1.0

@app.get('/health')
async def h(): return {'status': 'ok'}
@app.get('/v1/models')
async def lm(): return {'object': 'list', 'data': [{'id': 'model', 'object': 'model', 'created': 0}]}
@app.post('/v1/chat/completions')
async def cc(req: CR):
    try:
        msgs = [{'role': m.role, 'content': m.content} for m in req.messages]
        reply = gen(msgs, req.max_tokens or 512, req.temperature or 0.7, req.top_p or 0.9, req.top_k or 50, req.repetition_penalty or 1.0)
        return {'id': 'chatcmpl-0', 'object': 'chat.completion', 'model': 'model',
                'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': reply}, 'finish_reason': 'stop'}]}
    except Exception as e:
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        return JSONResponse(status_code=500, content={'error': str(e)})

threading.Thread(target=lambda: uvicorn.run(app, host=HOST, port=PORT, log_level='warning'), daemon=True).start()
print(f'Server: http://0.0.0.0:{PORT}')

In [ ]:
#@title 7. Gradio Chat UI (public link)
import gradio as gr, time

API = f'http://127.0.0.1:{PORT}/v1/chat/completions'

def chat_fn(msg, hist, temp, max_t, tp, tk, rp):
    msgs = []
    for u, a in hist:
        msgs += [{'role': 'user', 'content': u}, {'role': 'assistant', 'content': a}]
    msgs.append({'role': 'user', 'content': msg})
    try:
        import requests as r
        resp = r.post(API, json={'messages': msgs, 'temperature': temp, 'max_tokens': max_t, 'top_p': tp, 'top_k': tk, 'repetition_penalty': rp}, timeout=300)
        reply = resp.json()['choices'][0]['message']['content']
        partial = ''
        for w in reply.split(' '):
            partial += w + ' '
            yield partial
            time.sleep(0.015)
    except Exception as e:
        yield f'Error: {e}'

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
demo = gr.ChatInterface(fn=chat_fn, title='LLM Inference',
    description=f'Model: `{MODEL_PATH}` | GPU: `{gpu}`',
    additional_inputs=[
        gr.Slider(0, 2, value=0.7, step=0.05, label='Temp'),
        gr.Slider(64, 8192, value=2048, step=64, label='Max tokens'),
        gr.Slider(0, 1, value=0.9, step=0.05, label='Top-p'),
        gr.Slider(1, 200, value=50, step=1, label='Top-k'),
        gr.Slider(1, 2, value=1.0, step=0.05, label='Rep penalty'),
    ], fill_height=True)
demo.launch(share=True, server_name='0.0.0.0', server_port=7860, inline=False)

In [ ]:
#@title 8. Test API
!curl -s http://127.0.0.1:{PORT}/health

In [ ]:
#@title 9. Unload
import gc, torch
for x in ['model', 'tokenizer']:
    if x in dir(): del globals()[x]
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print('Freed.')